# SensorSpeak — Bosch Accelerometer Motion Event Explainer

**A fully local Python pipeline that ingests raw accelerometer CSV data, detects motion events with rule-based logic, generates plain-English summaries, and answers natural-language queries via LlamaIndex + Ollama.**

- No cloud • No API keys • No ML training • No deep learning
- Stack: Python 3.10 · pandas · numpy · matplotlib · LlamaIndex · Ollama (qwen2.5:0.5b) · bge-small-en-v1.5

---
| Section | Content |
|---|---|
| 0 | Install dependencies |
| 1 | Synthetic data generator |
| 2 | Optional CSV upload (Colab) |
| 3 | Normalize + feature engineering |
| 4 | Rule-based event detector |
| 5 | Plain-English summarizer |
| 6 | LlamaIndex vector index |
| 7 | Query engine |
| 8 | Visualization |
| 9 | Save outputs |
| 10 | Run pipeline + example queries |
| 11 | Interactive query cell |

## Section 0 — Install Dependencies

Run this cell once. In Colab the packages install to the runtime environment; locally use `pip install -r requirements.txt` instead.

In [ ]:
# Install all required packages (quiet mode suppresses verbose output)
import sys
!{sys.executable} -m pip install -q \
    pandas>=2.0.0 \
    numpy>=1.24.0 \
    matplotlib>=3.7.0 \
    llama-index-core>=0.10.0 \
    llama-index-llms-ollama>=0.1.0 \
    llama-index-embeddings-huggingface>=0.1.0 \
    sentence-transformers>=2.2.0
print('All packages installed successfully.')

## Section 1 — Synthetic Data Generator

Generates a 1900-sample (19 s at 100 Hz) accelerometer dataset across four labelled segments:
**idle → walking_like → impact → shaking**.

All segment parameters are named constants at the top of the cell — tune them freely.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend safe for Colab and headless envs
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass
from typing import List
import warnings
import os
import urllib.request
warnings.filterwarnings('ignore')

# ── DATA GENERATION CONSTANTS ─────────────────────────────────────────────────
SAMPLE_RATE_HZ        = 100        # samples per second
GRAVITY_Z             = 9.81       # m/s² gravity offset on Z axis

# Segment durations in seconds — change these to simulate longer/shorter events
SEG_IDLE_DURATION     = 5
SEG_WALK_DURATION     = 8
SEG_IMPACT_DURATION   = 2
SEG_SHAKE_DURATION    = 4

# Noise and signal amplitudes per segment
IDLE_NOISE            = 0.05       # very small random vibration
WALK_AMPLITUDE        = 1.2        # sinusoidal gait amplitude (m/s²)
WALK_FREQ_HZ          = 2.0        # ~2 steps per second
WALK_NOISE            = 0.2        # noise added on top of gait signal
IMPACT_SPIKE          = 12.0       # sharp spike magnitude (m/s²)
IMPACT_NOISE          = 0.5        # background noise during impact segment
SHAKE_AMPLITUDE       = 4.0        # large amplitude shaking (m/s²)
SHAKE_FREQ_HZ         = 8.0        # rapid oscillation frequency
SHAKE_NOISE           = 0.8        # noise added on top of shaking signal
RANDOM_SEED           = 42         # reproducibility seed


def generate_synthetic_data() -> pd.DataFrame:
    """Generate a 4-segment accelerometer dataset at SAMPLE_RATE_HZ."""
    rng = np.random.default_rng(RANDOM_SEED)
    segments = []

    def _make_time(duration: float) -> np.ndarray:
        n = int(duration * SAMPLE_RATE_HZ)
        return np.linspace(0, duration, n, endpoint=False)

    # ── Idle: device at rest, only tiny sensor noise ───────────────────────────
    t = _make_time(SEG_IDLE_DURATION)
    ax = rng.normal(0, IDLE_NOISE, len(t))
    ay = rng.normal(0, IDLE_NOISE, len(t))
    az = rng.normal(GRAVITY_Z, IDLE_NOISE, len(t))
    segments.append(pd.DataFrame({'accel_x': ax, 'accel_y': ay, 'accel_z': az,
                                   '_segment_label': 'idle'}))

    # ── Walking-like: sinusoidal gait pattern with noise ───────────────────────
    t = _make_time(SEG_WALK_DURATION)
    ax = WALK_AMPLITUDE * np.sin(2 * np.pi * WALK_FREQ_HZ * t) + rng.normal(0, WALK_NOISE, len(t))
    ay = (WALK_AMPLITUDE * 0.5) * np.cos(2 * np.pi * WALK_FREQ_HZ * t) + rng.normal(0, WALK_NOISE, len(t))
    az = rng.normal(GRAVITY_Z, WALK_NOISE, len(t))
    segments.append(pd.DataFrame({'accel_x': ax, 'accel_y': ay, 'accel_z': az,
                                   '_segment_label': 'walking_like'}))

    # ── Impact: brief baseline then a sharp mid-segment spike ──────────────────
    t = _make_time(SEG_IMPACT_DURATION)
    ax = rng.normal(0, IMPACT_NOISE, len(t))
    ay = rng.normal(0, IMPACT_NOISE, len(t))
    az = rng.normal(GRAVITY_Z, IMPACT_NOISE, len(t))
    mid = len(t) // 2                     # inject spike at midpoint
    ax[mid - 2 : mid + 3] += IMPACT_SPIKE
    az[mid - 2 : mid + 3] += IMPACT_SPIKE
    segments.append(pd.DataFrame({'accel_x': ax, 'accel_y': ay, 'accel_z': az,
                                   '_segment_label': 'impact'}))

    # ── Shaking: high-frequency oscillation on X and Y ────────────────────────
    t = _make_time(SEG_SHAKE_DURATION)
    ax = SHAKE_AMPLITUDE * np.sin(2 * np.pi * SHAKE_FREQ_HZ * t) + rng.normal(0, SHAKE_NOISE, len(t))
    ay = SHAKE_AMPLITUDE * np.cos(2 * np.pi * SHAKE_FREQ_HZ * t) + rng.normal(0, SHAKE_NOISE, len(t))
    az = rng.normal(GRAVITY_Z, SHAKE_NOISE, len(t))
    segments.append(pd.DataFrame({'accel_x': ax, 'accel_y': ay, 'accel_z': az,
                                   '_segment_label': 'shaking'}))

    df = pd.concat(segments, ignore_index=True)
    total_samples = len(df)
    df.insert(0, 'timestamp', np.arange(total_samples) / SAMPLE_RATE_HZ)
    return df


df_raw = generate_synthetic_data()
print(f'Generated {len(df_raw)} samples over {df_raw["timestamp"].iloc[-1]:.2f} s')
print(f'Columns: {list(df_raw.columns)}')
print()
print('Segment distribution:')
print(df_raw['_segment_label'].value_counts().to_string())
print()
print(df_raw.head(3).to_string())

## Section 2 — Optional CSV Upload (Colab)

To use **real sensor data** instead of the synthetic dataset:
1. Uncomment the upload block below
2. Upload a CSV with columns: `timestamp`, `accel_x`, `accel_y`, `accel_z`
3. Comment out the `df_raw = generate_synthetic_data()` call in Section 1

> Timestamps should be in seconds (float). `accel_z` should include gravity (~9.81 m/s² at rest).

In [ ]:
# ── OPTIONAL: Upload a real CSV in Google Colab ────────────────────────────────
# Uncomment the block below to enable file upload.
# Required columns: timestamp (s), accel_x, accel_y, accel_z (all float, m/s²)

# from google.colab import files as colab_files
#
# def load_uploaded_csv() -> pd.DataFrame:
#     print('Please upload your accelerometer CSV file...')
#     uploaded = colab_files.upload()
#     filename = list(uploaded.keys())[0]
#     df = pd.read_csv(filename)
#     required_cols = {'timestamp', 'accel_x', 'accel_y', 'accel_z'}
#     missing = required_cols - set(df.columns)
#     if missing:
#         raise ValueError(f'CSV is missing required columns: {missing}')
#     print(f'Loaded {len(df)} rows from "{filename}"')
#     return df
#
# df_raw = load_uploaded_csv()   # ← uncomment this line and comment Section 1 call

print('Section 2: CSV upload is commented out — using synthetic data from Section 1.')
print(f'Current df_raw shape: {df_raw.shape}')

## Section 3 — Normalize + Feature Engineering

- **Z-score normalize** `accel_x`, `accel_y`, `accel_z` — raw values preserved in `_raw_*` columns
- **accel_magnitude** = √(x² + y² + z²) — captures total motion energy
- **rolling_mean** and **rolling_std** over a 50-sample window (~0.5 s) — smooths spikes and reveals trends

In [ ]:
# ── SIGNAL PROCESSING CONSTANTS ───────────────────────────────────────────────
ROLLING_WINDOW  = 50     # samples; 50 / 100Hz = 0.5 s — increase to smooth more
ZSCORE_EPS      = 1e-8   # prevent division by zero when std ≈ 0


def normalize_and_engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Z-score normalize axes, compute resultant magnitude and rolling statistics."""
    required = {'accel_x', 'accel_y', 'accel_z'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Input DataFrame is missing required columns: {missing}')

    df = df.copy()

    # Z-score each axis; preserve originals for the visualization
    for axis in ('accel_x', 'accel_y', 'accel_z'):
        df[f'_raw_{axis}'] = df[axis]
        mean_val = df[axis].mean()
        std_val  = df[axis].std()
        df[axis] = (df[axis] - mean_val) / (std_val + ZSCORE_EPS)

    # Resultant magnitude across all three normalized axes
    df['accel_magnitude'] = np.sqrt(
        df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2
    )

    # Rolling statistics — center=True gives a symmetric window
    df['rolling_mean'] = (
        df['accel_magnitude']
        .rolling(window=ROLLING_WINDOW, min_periods=1, center=True)
        .mean()
    )
    df['rolling_std'] = (
        df['accel_magnitude']
        .rolling(window=ROLLING_WINDOW, min_periods=1, center=True)
        .std()
        .fillna(0)
    )

    return df


df = normalize_and_engineer_features(df_raw)
print(f'Feature engineering complete.')
print(f'New columns: {[c for c in df.columns if c not in df_raw.columns]}')
print()
print(df[['timestamp', 'accel_magnitude', 'rolling_mean', 'rolling_std']].describe().round(4).to_string())

## Section 4 — Rule-Based Event Detector

Each sample is labelled using deterministic threshold rules on `rolling_mean` and `rolling_std`. Consecutive same-label runs of ≥ `MIN_EVENT_SAMPLES` are consolidated into `MotionEvent` objects.

| Event | Rule |
|---|---|
| idle | rolling_std < 0.15 |
| impact | rolling_mean > 2.5 AND rolling_std > 1.0 |
| shaking | rolling_std > 1.8 |
| walking | 0.8 ≤ rolling_mean ≤ 2.5 AND 0.10 ≤ rolling_std ≤ 1.8 |
| unknown | none of the above |

In [ ]:
# ── EVENT DETECTION THRESHOLDS ────────────────────────────────────────────────
IDLE_STD_MAX        = 0.15   # rolling_std must be below this for idle
IMPACT_MEAN_MIN     = 2.5    # rolling_mean must exceed this for impact
IMPACT_STD_MIN      = 1.0    # rolling_std must exceed this for impact
SHAKING_STD_MIN     = 1.8    # rolling_std must exceed this for shaking
WALKING_MEAN_MIN    = 0.8    # rolling_mean lower bound for walking
WALKING_MEAN_MAX    = 2.5    # rolling_mean upper bound for walking
WALKING_STD_MIN     = 0.10   # rolling_std lower bound for walking
WALKING_STD_MAX     = 1.8    # rolling_std upper bound for walking
MIN_EVENT_SAMPLES   = 10     # minimum consecutive same-label samples to register an event


@dataclass
class MotionEvent:
    start:    float   # start timestamp (s)
    end:      float   # end timestamp (s)
    type:     str     # event category string
    max_mag:  float   # peak accel_magnitude in the event window
    mean_mag: float   # mean accel_magnitude in the event window
    seed:     str     # short descriptor string fed to the summarizer


def _classify_sample(rmean: float, rstd: float) -> str:
    """Apply threshold rules to one (rolling_mean, rolling_std) pair."""
    if rstd < IDLE_STD_MAX:
        return 'idle'
    if rmean > IMPACT_MEAN_MIN and rstd > IMPACT_STD_MIN:
        return 'impact'
    if rstd > SHAKING_STD_MIN:
        return 'shaking'
    if WALKING_MEAN_MIN <= rmean <= WALKING_MEAN_MAX and WALKING_STD_MIN <= rstd <= WALKING_STD_MAX:
        return 'walking'
    return 'unknown'


# Human-readable seeds that the summarizer appends to each event description
_SEED_MAP = {
    'idle':    'minimal vibration; device appears to be at rest',
    'walking': 'rhythmic oscillation consistent with footsteps or a rotating shaft',
    'impact':  'sharp high-amplitude spike; possible drop, collision, or hammer blow',
    'shaking': 'rapid high-frequency oscillation indicating intentional agitation or loose component',
    'unknown': 'unclassified motion pattern; threshold tuning may be required',
}


def detect_events(df: pd.DataFrame) -> List[MotionEvent]:
    """Segment the DataFrame into MotionEvent objects."""
    for col in ('timestamp', 'rolling_mean', 'rolling_std', 'accel_magnitude'):
        if col not in df.columns:
            raise ValueError(f'Required column "{col}" is missing from DataFrame.')

    # Classify every sample
    labels = [
        _classify_sample(rm, rs)
        for rm, rs in zip(df['rolling_mean'], df['rolling_std'])
    ]

    events: List[MotionEvent] = []
    i, n = 0, len(labels)

    while i < n:
        current_label = labels[i]
        j = i
        # extend the run as long as the label is identical
        while j < n and labels[j] == current_label:
            j += 1
        run_length = j - i
        # only register runs long enough to be meaningful
        if run_length >= MIN_EVENT_SAMPLES:
            window = df.iloc[i:j]
            events.append(MotionEvent(
                start    = float(window['timestamp'].iloc[0]),
                end      = float(window['timestamp'].iloc[-1]),
                type     = current_label,
                max_mag  = float(window['accel_magnitude'].max()),
                mean_mag = float(window['accel_magnitude'].mean()),
                seed     = _SEED_MAP[current_label],
            ))
        i = j

    return events


events = detect_events(df)
print(f'Detected {len(events)} events:')
print(f'{"Start":>8}  {"End":>8}  {"Type":<14}  {"MaxMag":>8}  {"MeanMag":>8}')
print('-' * 58)
for ev in events:
    print(f'{ev.start:>8.2f}  {ev.end:>8.2f}  {ev.type:<14}  {ev.max_mag:>8.4f}  {ev.mean_mag:>8.4f}')

## Section 5 — Plain-English Summarizer

Converts each `MotionEvent` into a single human-readable sentence including:
- Time window and duration
- Severity label (low / moderate / high / severe)
- Peak and mean magnitude
- Explanation seed from the detection rule

In [ ]:
def _severity_label(max_mag: float) -> str:
    """Map peak magnitude to a human-readable severity tier."""
    if max_mag < 1.5:
        return 'low'
    if max_mag < 3.5:
        return 'moderate'
    if max_mag < 7.0:
        return 'high'
    return 'severe'


def summarize_event(event: MotionEvent) -> str:
    """Return a single-sentence plain-English description of a MotionEvent."""
    duration = event.end - event.start
    sev = _severity_label(event.max_mag)
    return (
        f'From {event.start:.2f}s to {event.end:.2f}s ({duration:.2f}s), '
        f'a {sev}-severity "{event.type}" event was detected '
        f'(peak magnitude {event.max_mag:.4f}, mean {event.mean_mag:.4f} m/s\u00b2) '
        f'\u2014 {event.seed}.'
    )


summaries = [summarize_event(ev) for ev in events]
print(f'Generated {len(summaries)} plain-English summaries:')
print()
for i, s in enumerate(summaries, 1):
    print(f'  [{i}] {s}')

## Section 6 — LlamaIndex Vector Index

Each event summary becomes a `Document` in a `VectorStoreIndex`:
- **Embeddings**: `bge-small-en-v1.5` via HuggingFace (local, no API key)
- **LLM**: `qwen2.5:0.5b` via Ollama (local daemon, ~350 MB one-time download)
- **Fallback**: keyword matcher if Ollama is not running

### Ollama Setup in Colab
Uncomment `install_ollama_colab()` at the bottom of this cell to install Ollama automatically. This takes ~5 minutes on the first run.

In [ ]:
import subprocess
import time

# ── LLM / INDEX SETTINGS ──────────────────────────────────────────────────────
OLLAMA_MODEL       = 'qwen2.5:0.5b'          # local LLM served by Ollama
OLLAMA_TIMEOUT     = 180                      # seconds before a query times out
EMBED_MODEL_NAME   = 'BAAI/bge-small-en-v1.5' # HuggingFace sentence embedding
SIMILARITY_TOP_K   = 4                        # documents returned per query
RESPONSE_MODE      = 'compact'                # LlamaIndex synthesis mode

# ── Import LlamaIndex (graceful fallback if not installed) ────────────────────
LLAMAINDEX_AVAILABLE = False
try:
    from llama_index.core import VectorStoreIndex, Document, Settings
    from llama_index.llms.ollama import Ollama
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    LLAMAINDEX_AVAILABLE = True
    print('LlamaIndex imported successfully.')
except ImportError as e:
    print(f'LlamaIndex import failed ({e}) — will use keyword fallback.')


def install_ollama_colab() -> None:
    """Download and start Ollama inside a Google Colab runtime."""
    print('Downloading Ollama installer...')
    subprocess.run(['curl', '-fsSL', 'https://ollama.ai/install.sh', '-o', '/tmp/install.sh'],
                   check=True, capture_output=True)
    print('Installing Ollama...')
    subprocess.run(['sh', '/tmp/install.sh'], check=True, capture_output=True)
    print('Starting Ollama daemon...')
    subprocess.Popen(['ollama', 'serve'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)  # give the daemon time to start accepting connections
    print(f'Pulling model "{OLLAMA_MODEL}" (may take a few minutes)...')
    subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
    print(f'Ollama ready with model "{OLLAMA_MODEL}".')


# ── Uncomment the line below to install Ollama automatically in Colab ─────────
# install_ollama_colab()


def _is_ollama_running() -> bool:
    """Return True if the local Ollama daemon responds on port 11434."""
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=3)
        return True
    except Exception:
        return False


def _build_documents(events: List[MotionEvent], summaries: List[str]) -> list:
    """Wrap each event summary in a LlamaIndex Document with searchable metadata."""
    docs = []
    for ev, summary in zip(events, summaries):
        docs.append(Document(
            text=summary,
            metadata={
                'event_type': ev.type,
                'start':      ev.start,
                'end':        ev.end,
                'max_mag':    round(ev.max_mag, 4),
            }
        ))
    return docs


def build_index(events: List[MotionEvent], summaries: List[str]):
    """Build a VectorStoreIndex, or return None to fall back to keyword search."""
    if not LLAMAINDEX_AVAILABLE:
        print('LlamaIndex unavailable — using keyword fallback.')
        return None

    if not _is_ollama_running():
        print('Ollama daemon not detected on localhost:11434 — using keyword fallback.')
        print('Tip: uncomment install_ollama_colab() above and re-run this cell.')
        return None

    try:
        print('Configuring LlamaIndex settings...')
        Settings.llm = Ollama(model=OLLAMA_MODEL, request_timeout=OLLAMA_TIMEOUT)
        Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
        docs = _build_documents(events, summaries)
        print(f'Building VectorStoreIndex from {len(docs)} documents...')
        index = VectorStoreIndex.from_documents(docs)
        print(f'Index ready. Embedding model: {EMBED_MODEL_NAME}  |  LLM: {OLLAMA_MODEL}')
        return index
    except Exception as exc:
        print(f'Index build failed: {exc}')
        print('Falling back to keyword matcher.')
        return None


index = build_index(events, summaries)

## Section 7 — Query Engine

`query_events(question)` is the unified interface:
- **LlamaIndex path**: semantic retrieval + Ollama LLM synthesis (when Ollama is running)
- **Keyword fallback**: term-overlap scoring over summaries (always available)

Set `verbose=True` to see how many source documents were retrieved.

In [ ]:
def _keyword_fallback(question: str, summaries: List[str]) -> str:
    """Score summaries by keyword overlap and return the top matches."""
    q_terms = question.lower().split()
    scored = []
    for s in summaries:
        score = sum(term in s.lower() for term in q_terms)
        if score > 0:
            scored.append((score, s))
    if not scored:
        return 'No matching events found for your query. Try keywords like: idle, impact, shaking, walking.'
    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[:SIMILARITY_TOP_K]
    header = '[Keyword fallback — Ollama not running]\n'
    return header + '\n'.join(f'  \u2022 {s}' for _, s in top)


def query_events(question: str, verbose: bool = False) -> str:
    """Query motion events with a natural-language question."""
    if not question.strip():
        return 'Please provide a non-empty question.'

    # Attempt LlamaIndex + Ollama path first
    if index is not None:
        try:
            qe = index.as_query_engine(
                similarity_top_k=SIMILARITY_TOP_K,
                response_mode=RESPONSE_MODE,
            )
            response = qe.query(question)
            if verbose:
                print(f'[LlamaIndex] {len(response.source_nodes)} source node(s) retrieved.')
            return str(response)
        except Exception as exc:
            if verbose:
                print(f'LLM query error ({exc}); falling back to keyword matcher.')

    # Keyword fallback — always available
    return _keyword_fallback(question, summaries)


# ── Three built-in example queries ────────────────────────────────────────────
EXAMPLE_QUERIES = [
    'Were there any abnormal or high-severity events?',
    'How long did the device remain idle?',
    'Describe any impact events detected.',
]

print('=' * 65)
print('EXAMPLE QUERY RESULTS')
print('=' * 65)
for q in EXAMPLE_QUERIES:
    print(f'\nQ: {q}')
    print(f'A: {query_events(q)}')
    print('-' * 65)

## Section 8 — Visualization

Three-panel matplotlib figure:
- **Panel 1**: Raw accelerometer axes (X, Y, Z) over time
- **Panel 2**: Resultant magnitude with colour-coded event bands
- **Panel 3**: Rolling mean and rolling standard deviation

Saved to `outputs/accelerometer_overview.png`.

In [ ]:
# ── VISUALIZATION CONSTANTS ───────────────────────────────────────────────────
VIZ_OUTPUT_FILE  = os.path.join('outputs', 'accelerometer_overview.png')
VIZ_DPI          = 150
VIZ_FIGSIZE      = (14, 10)

EVENT_COLORS = {
    'idle':    '#90EE90',   # light green
    'walking': '#87CEEB',   # sky blue
    'impact':  '#FF6B6B',   # coral red
    'shaking': '#FFD700',   # gold
    'unknown': '#D3D3D3',   # light gray
}


def _shade_event_bands(ax, events: List[MotionEvent]) -> None:
    """Fill background spans for each detected event window."""
    for ev in events:
        ax.axvspan(
            ev.start, ev.end,
            alpha=0.25,
            color=EVENT_COLORS.get(ev.type, '#D3D3D3'),
        )


def visualize(df: pd.DataFrame, events: List[MotionEvent],
              output_path: str = VIZ_OUTPUT_FILE) -> None:
    """Render the 3-panel overview figure and save to disk."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    t = df['timestamp']

    fig, axes = plt.subplots(3, 1, figsize=VIZ_FIGSIZE, sharex=True)
    fig.suptitle('SensorSpeak — Accelerometer Overview', fontsize=14, fontweight='bold')

    # ── Panel 1: raw axes ─────────────────────────────────────────────────────
    ax1 = axes[0]
    ax1.plot(t, df['_raw_accel_x'], color='#E74C3C', linewidth=0.7, label='X raw')
    ax1.plot(t, df['_raw_accel_y'], color='#2ECC71', linewidth=0.7, label='Y raw')
    ax1.plot(t, df['_raw_accel_z'], color='#3498DB', linewidth=0.7, label='Z raw')
    ax1.set_ylabel('Acceleration (m/s\u00b2)')
    ax1.set_title('Panel 1: Raw Accelerometer Axes')
    ax1.legend(loc='upper right', fontsize=8)
    ax1.grid(True, alpha=0.3)

    # ── Panel 2: magnitude + coloured event bands ─────────────────────────────
    ax2 = axes[1]
    _shade_event_bands(ax2, events)
    ax2.plot(t, df['accel_magnitude'], color='#8E44AD', linewidth=0.8, label='Magnitude')
    ax2.set_ylabel('Magnitude (normalised)')
    ax2.set_title('Panel 2: Resultant Magnitude + Detected Event Bands')
    # Build a clean legend with one patch per event type
    event_patches = [
        mpatches.Patch(color=c, alpha=0.5, label=k)
        for k, c in EVENT_COLORS.items()
    ]
    mag_line = plt.Line2D([0], [0], color='#8E44AD', linewidth=1.5, label='Magnitude')
    ax2.legend(handles=event_patches + [mag_line], loc='upper right', fontsize=7, ncol=3)
    ax2.grid(True, alpha=0.3)

    # ── Panel 3: rolling statistics ───────────────────────────────────────────
    ax3 = axes[2]
    ax3.plot(t, df['rolling_mean'], color='#F39C12', linewidth=1.0, label='Rolling mean')
    ax3.plot(t, df['rolling_std'],  color='#1ABC9C', linewidth=1.0, label='Rolling std')
    # Annotate threshold lines so readers can see where detections fire
    ax3.axhline(IDLE_STD_MAX, color='#90EE90', linestyle='--', linewidth=0.8, alpha=0.7, label=f'Idle std max ({IDLE_STD_MAX})')
    ax3.axhline(IMPACT_MEAN_MIN, color='#FF6B6B', linestyle='--', linewidth=0.8, alpha=0.7, label=f'Impact mean min ({IMPACT_MEAN_MIN})')
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Value')
    ax3.set_title('Panel 3: Rolling Mean and Std of Magnitude (with threshold lines)')
    ax3.legend(loc='upper right', fontsize=7, ncol=2)
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_path, dpi=VIZ_DPI, bbox_inches='tight')
    plt.show()
    print(f'Visualization saved to "{output_path}".')


visualize(df, events)

## Section 9 — Save Outputs

Saves two CSV files to `outputs/`:
- `synthetic_accel_data.csv` — full feature-engineered DataFrame
- `detected_events.csv` — one row per event with plain-English summary column

In [ ]:
# ── OUTPUT FILE PATHS ─────────────────────────────────────────────────────────
OUTPUT_DIR       = 'outputs'
CSV_RAW_FILE     = os.path.join(OUTPUT_DIR, 'synthetic_accel_data.csv')
CSV_EVENTS_FILE  = os.path.join(OUTPUT_DIR, 'detected_events.csv')


def save_outputs(df: pd.DataFrame, events: List[MotionEvent],
                 summaries: List[str]) -> None:
    """Persist the feature DataFrame and event table to CSV files."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Full sensor + feature data
    df.to_csv(CSV_RAW_FILE, index=False)
    print(f'Raw data saved   -> {CSV_RAW_FILE}  ({len(df)} rows, {len(df.columns)} columns)')

    # Event table with summaries
    records = []
    for ev, summary in zip(events, summaries):
        records.append({
            'start_s':    round(ev.start, 3),
            'end_s':      round(ev.end, 3),
            'duration_s': round(ev.end - ev.start, 3),
            'event_type': ev.type,
            'max_mag':    round(ev.max_mag, 4),
            'mean_mag':   round(ev.mean_mag, 4),
            'summary':    summary,
        })
    df_events = pd.DataFrame(records)
    df_events.to_csv(CSV_EVENTS_FILE, index=False)
    print(f'Events saved     -> {CSV_EVENTS_FILE}  ({len(df_events)} rows)')
    print()
    print(df_events[['start_s', 'end_s', 'event_type', 'max_mag', 'summary']].to_string(index=False))


save_outputs(df, events, summaries)

## Section 10 — Pipeline Summary

A concise end-to-end summary of what the pipeline produced.

In [ ]:
def print_pipeline_summary(
    df: pd.DataFrame,
    events: List[MotionEvent],
    summaries: List[str],
    index_built: bool,
) -> None:
    """Print a formatted end-to-end pipeline summary."""
    print('=' * 65)
    print('SENSOSPEAK PIPELINE SUMMARY')
    print('=' * 65)
    print(f'  Samples generated  : {len(df):,}')
    print(f'  Duration           : {df["timestamp"].iloc[-1]:.2f} s')
    print(f'  Sample rate        : {SAMPLE_RATE_HZ} Hz')
    print(f'  Features           : {[c for c in df.columns if not c.startswith("_") and c != "timestamp"]}')
    print()
    print(f'  Events detected    : {len(events)}')
    type_counts = {}
    for ev in events:
        type_counts[ev.type] = type_counts.get(ev.type, 0) + 1
    for etype, cnt in sorted(type_counts.items()):
        print(f'    {etype:<14}: {cnt}')
    print()
    print(f'  Index backend      : {"LlamaIndex + Ollama (" + OLLAMA_MODEL + ")" if index_built else "Keyword fallback"}')
    print(f'  Embedding model    : {EMBED_MODEL_NAME if index_built else "N/A"}')
    print()
    print(f'  Output files:')
    print(f'    {CSV_RAW_FILE}')
    print(f'    {CSV_EVENTS_FILE}')
    print(f'    {VIZ_OUTPUT_FILE}')
    print('=' * 65)


print_pipeline_summary(df, events, summaries, index is not None)

## Section 11 — Interactive Query

Edit `MY_QUESTION` below and run this cell to ask any natural-language question about the detected motion events.

**Example questions:**
- `'Were there any severe or dangerous events?'`
- `'How long did the device remain stationary?'`
- `'What happened between 13 and 15 seconds?'`
- `'Were there any walking or rhythmic patterns?'`
- `'Summarize all events in order.'`

In [ ]:
# ── Edit this question and run the cell ───────────────────────────────────────
MY_QUESTION = 'Were there any abnormal or high-severity events?'

print(f'Q: {MY_QUESTION}')
print()
answer = query_events(MY_QUESTION, verbose=True)
print(f'A: {answer}')